In [1]:
from pathlib import Path
import importlib.util
import json
import joblib
import pandas as pd
from flask import Flask, request, jsonify

In [2]:
from pathlib import Path

ROOT = Path.cwd().parent
DEPLOY = ROOT / "artifacts" / "deployment"
MODEL_PATH = DEPLOY / "fake_profile_detector.pkl"
FEATURES_PATH = DEPLOY / "model_features.pkl"
THRESHOLDS_PATH = DEPLOY / "risk_thresholds.json"
EXPLANATION_PATH = DEPLOY / "explanation.py"


In [3]:
# Load model with diagnostics; prefer matching kernel with scikit-learn 1.7.1
import traceback
print("NOTE: If you installed scikit-learn 1.7.1, ensure the notebook kernel uses that Python environment before re-running this cell.")

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Model not found at {MODEL_PATH}")

print('Model path exists, size (bytes):', MODEL_PATH.stat().st_size)
try:
    import sklearn
    print('sklearn version (runtime):', sklearn.__version__)
except Exception:
    print('Could not import sklearn to show version')

try:
    model = joblib.load(MODEL_PATH)
    print('Model loaded successfully:', type(model))
except Exception as e:
    print('Failed to load model with joblib:', repr(e))
    traceback.print_exc()
    # Show a small sample of the file header to help debugging
    try:
        with open(MODEL_PATH, 'rb') as fh:
            head = fh.read(256)
        print('File header (utf-8 safe):')
        print(head.decode('utf-8', errors='replace'))
    except Exception as e2:
        print('Could not read file header:', e2)
    print('\nHints:')
    print('- Kernel may not match the Python environment where scikit-learn 1.7.1 is installed.')
    print('- Switch kernel to the Python interpreter that has scikit-learn 1.7.1, then re-run this cell.')
    print('- Or re-generate the model artifact using the current environment or convert it to a portable format (ONNX).')
    print('Continuing without raising; `model` may be undefined if load failed.')

NOTE: If you installed scikit-learn 1.7.1, ensure the notebook kernel uses that Python environment before re-running this cell.
Model path exists, size (bytes): 3093481
sklearn version (runtime): 1.7.1
Model loaded successfully: <class 'sklearn.ensemble._forest.RandomForestClassifier'>


In [4]:
# Load features, thresholds, explanation module, define helpers and Flask app
if not FEATURES_PATH.exists():
    raise FileNotFoundError(f"Features file not found at {FEATURES_PATH}")
features = joblib.load(FEATURES_PATH)

if not THRESHOLDS_PATH.exists():
    raise FileNotFoundError(f"Thresholds file not found at {THRESHOLDS_PATH}")
with open(THRESHOLDS_PATH, "r") as fh:
    thresholds = json.load(fh)

if not EXPLANATION_PATH.exists():
    explanation = None
else:
    spec = importlib.util.spec_from_file_location("explanation", EXPLANATION_PATH)
    explanation = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(explanation)

def prepare_row(payload):
    df = pd.DataFrame([payload])
    # Ensure columns are in the expected order and missing columns are filled with NaN
    for col in features:
        if col not in df.columns:
            df[col] = pd.NA
    # Reorder to model features
    df = df[features]
    # Coerce numeric where possible and fill NaNs with 0 for inference
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.fillna(0)
    return df

def score_row(df_row):
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(df_row)[:, 1]
        score = int(float(proba[0]) * 100)
    else:
        pred = model.predict(df_row)[0]
        score = int(pred * 100)

    low_thr = int(thresholds.get("low", 30))
    med_thr = int(thresholds.get("medium", 70))

    if score >= med_thr:
        label = "High Risk"
    elif score >= low_thr:
        label = "Medium Risk"
    else:
        label = "Low Risk"

    return score, label

app = Flask(__name__)

@app.route('/predict', methods=['POST'])
def predict():
    payload = request.get_json()
    if payload is None:
        return jsonify({"error": "Invalid JSON payload"}), 400

    try:
        row = prepare_row(payload)
        score, label = score_row(row)
    except Exception as e:
        return jsonify({"error": "Scoring failed", "detail": str(e)}), 500

    reasons = []
    try:
        if explanation and hasattr(explanation, 'generate_reasons'):
            reasons = explanation.generate_reasons(row.iloc[0].to_dict())
    except Exception:
        reasons = []

    return jsonify({
        "risk_score": score,
        "status": label,
        "reasons": reasons
    })

# If running inside the notebook interactively, you can start the app with:
# from werkzeug.serving import run_simple
# run_simple('0.0.0.0', 5000, app)


In [5]:
from werkzeug.serving import run_simple

run_simple('0.0.0.0', 5000, app)

 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.16.15.215:5000
Press CTRL+C to quit
